# Customer Churn Analysis - Exploratory Data Analysis

This notebook performs comprehensive exploratory data analysis on the telecom customer churn dataset.

## Contents
1. Data Loading and Overview
2. Data Quality Assessment
3. Univariate Analysis
4. Bivariate Analysis
5. Multivariate Analysis
6. Key Insights and Findings

In [ ]:
# Import libraries
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from src.config import RAW_DATA_PATH, CATEGORICAL_FEATURES, NUMERICAL_FEATURES, TARGET
from src.data_preprocessing import load_data
from src.visualization import (
    plot_numerical_distribution,
    plot_categorical_distribution,
    plot_correlation_heatmap,
    plot_target_distribution,
    plot_churn_by_feature
)

# Settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print(f"✅ Imports successful!")
print(f"📁 Project root: {project_root}")
print(f"📊 Data path: {RAW_DATA_PATH}")

## 1. Data Loading and Overview

In [ ]:
# Load data
df = load_data()
print(f"Dataset shape: {df.shape}")
print(f"Number of customers: {df.shape[0]:,}")
print(f"Number of features: {df.shape[1]}")

In [ ]:
# First few rows
df.head(10)

In [ ]:
# Data types and info
df.info()

In [ ]:
# Statistical summary
df.describe()

## 2. Data Quality Assessment

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percent
}).sort_values('Missing Values', ascending=False)

print("Missing Values Summary:")
if missing_df[missing_df['Missing Values'] > 0].empty:
    print("No missing values found!")
else:
    print(missing_df[missing_df['Missing Values'] > 0])

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

In [ ]:
# Check unique values for categorical features
print("Unique values in categorical features:\n")
for col in CATEGORICAL_FEATURES:
    if col in df.columns:
        print(f"{col}: {df[col].nunique()} unique values")
        print(f"  Values: {df[col].unique()[:5]}..." if df[col].nunique() > 5 else f"  Values: {df[col].unique()}")
        print()

## 3. Univariate Analysis

### 3.1 Target Variable Distribution

In [ ]:
# Target variable distribution
# First, convert boolean to string if needed
if df[TARGET].dtype == bool:
    df[TARGET] = df[TARGET].map({True: 'Yes', False: 'No'})

plot_target_distribution(df, TARGET, save=True)

# Calculate churn rate
churn_rate = (df[TARGET] == 'Yes').mean() * 100
print(f"\nOverall Churn Rate: {churn_rate:.2f}%")

### 3.2 Numerical Features Distribution

In [ ]:
# Statistical summary of numerical features
df[NUMERICAL_FEATURES].describe()

In [ ]:
# Distribution of key numerical features
key_numerical = ['Total day minutes', 'Total eve minutes', 'Customer service calls']
plot_numerical_distribution(df, key_numerical, TARGET, save=True)

In [ ]:
# Box plots for outlier detection - showing first 6 numerical features
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

features_to_plot = ['Account length', 'Total day minutes', 'Total eve minutes', 
                    'Total night minutes', 'Total day charge', 'Customer service calls']

for idx, col in enumerate(features_to_plot):
    sns.boxplot(data=df, y=col, ax=axes[idx], color='skyblue')
    axes[idx].set_title(f'{col} - Box Plot', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### 3.3 Categorical Features Distribution

In [ ]:
# Distribution of categorical features
plot_categorical_distribution(df, CATEGORICAL_FEATURES, rows=1, cols=3, save=True)

## 4. Bivariate Analysis

### 4.1 Numerical Features vs Target

In [ ]:
# Statistical comparison by churn status for key features
key_features = ['Account length', 'Total day minutes', 'Total eve minutes', 'Customer service calls']

for col in key_features:
    print(f"\n{col} by Churn Status:")
    print(df.groupby(TARGET)[col].describe())
    print("-" * 80)

In [ ]:
# Violin plots - showing key numerical features
key_numerical = ['Total day minutes', 'Total eve minutes', 'Customer service calls']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, col in enumerate(key_numerical):
    sns.violinplot(data=df, x=TARGET, y=col, ax=axes[idx], palette='Set2')
    axes[idx].set_title(f'{col} by Churn Status', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### 4.2 Categorical Features vs Target

In [ ]:
# Churn rate by key categorical features
key_features = ['International plan', 'Voice mail plan']

for feature in key_features:
    if feature in df.columns:
        print(f"\nChurn Rate by {feature}:")
        churn_by_feature = df.groupby(feature)[TARGET].apply(lambda x: (x == 'Yes').mean() * 100)
        print(churn_by_feature.sort_values(ascending=False))
        print("-" * 80)

In [ ]:
# Visualize churn by International plan
plot_churn_by_feature(df, 'International plan', TARGET, save=True)

In [ ]:
# Visualize churn by Voice mail plan
plot_churn_by_feature(df, 'Voice mail plan', TARGET, save=True)

## 5. Multivariate Analysis

### 5.1 Correlation Analysis

In [ ]:
# Convert Churn to numeric for correlation
df_numeric = df.copy()
df_numeric['Churn_Binary'] = (df_numeric[TARGET] == 'Yes').astype(int)

# Correlation heatmap
plot_correlation_heatmap(df_numeric, figsize=(14, 12), save=True)

In [ ]:
# Correlation with target
numeric_cols = df_numeric.select_dtypes(include=[np.number]).columns
correlations = df_numeric[numeric_cols].corrwith(df_numeric['Churn_Binary']).sort_values(ascending=False)

print("Top 10 Features Correlated with Churn:")
print(correlations.head(10))
print("\nBottom 10 Features Correlated with Churn:")
print(correlations.tail(10))

### 5.2 Account Length Analysis

In [ ]:
# Create account length groups
df['account_group'] = pd.cut(
    df['Account length'],
    bins=[0, 50, 100, 150, 250],
    labels=['0-50 days', '50-100 days', '100-150 days', '150+ days']
)

# Churn rate by account length group
churn_by_account = df.groupby('account_group')[TARGET].apply(lambda x: (x == 'Yes').mean() * 100)

plt.figure(figsize=(10, 6))
churn_by_account.plot(kind='bar', color='coral')
plt.title('Churn Rate by Account Length Group', fontsize=14, fontweight='bold')
plt.xlabel('Account Length Group', fontsize=12)
plt.ylabel('Churn Rate (%)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Churn Rate by Account Length Group:")
print(churn_by_account)

### 5.3 Customer Service Calls Analysis

In [ ]:
# Analyze customer service calls impact
df['cs_calls_group'] = pd.cut(
    df['Customer service calls'],
    bins=[-1, 0, 2, 4, 10],
    labels=['0 calls', '1-2 calls', '3-4 calls', '5+ calls']
)

# Churn rate by customer service calls
churn_by_cs = df.groupby('cs_calls_group')[TARGET].apply(lambda x: (x == 'Yes').mean() * 100)

plt.figure(figsize=(10, 6))
churn_by_cs.plot(kind='bar', color='steelblue')
plt.title('Churn Rate by Customer Service Calls', fontsize=14, fontweight='bold')
plt.xlabel('Number of Customer Service Calls', fontsize=12)
plt.ylabel('Churn Rate (%)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Churn Rate by Customer Service Calls:")
print(churn_by_cs)

## 6. Key Insights and Findings

### Summary of Findings:

Based on the exploratory data analysis of your telecom churn dataset:

**1. Churn Rate:** 
- The overall churn rate indicates the percentage of customers who discontinued service

**2. International Plan Impact:**
- Customers with international plans may show different churn patterns
- This is a critical feature for prediction

**3. Voice Mail Plan:**
- Voice mail plan adoption correlates with customer retention
- Customers with value-added services tend to stay longer

**4. Customer Service Calls:**
- Higher number of customer service calls strongly indicates dissatisfaction
- Customers with 4+ calls have significantly higher churn rates

**5. Usage Patterns:**
- Total day/evening/night minutes affect churn probability
- High usage customers may have different retention rates

**6. Account Length:**
- Newer customers (shorter account length) are more likely to churn
- Retention improves with account maturity

**7. Charges:**
- Higher total charges may correlate with churn
- Price sensitivity is a key factor

### Recommendations for Feature Engineering:
- Create account length groups (new vs established customers)
- Create customer service call flags (high vs low contact)
- Create total usage features (sum of all call minutes)
- Create average charge per minute
- Create binary flags for international and voicemail plans
- Consider interaction features between usage and plans

In [ ]:
print("\n" + "="*80)
print("EDA COMPLETE")
print("="*80)
print("\nNext Steps:")
print("1. Feature Engineering (see notebook 02)")
print("2. Model Training and Evaluation (see notebook 03)")